In [1]:
from pyspark.sql.functions import row_number
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    lag, col, lit, radians, sin, 
    cos, asin, sqrt, unix_timestamp, abs, min
)
from pyspark.errors.exceptions.base import PySparkRuntimeError
from pyspark.sql import functions as F
from pyspark.sql.column import Column
from pyspark.sql.window import Window
from dataclasses import dataclass
from pyspark import SparkContext
from datetime import datetime
from pyspark.sql.types import *
import numpy as np
import pandas as pd
import os


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/conda/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/conda/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/opt/conda/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/opt/conda/lib/python3.11/site-packages/tornado

AttributeError: _ARRAY_API not found

In [2]:
@dataclass
class Point:
    latitude: np.float64
    longitude: np.float64
    
    @property
    def point(self) -> tuple:
        return (
            self.latitude,
            self.longitude
        )

In [3]:

# For detecting a moving ship
SOG_MOVE = 1

# 50-nautical-mile for in radius checking
NAUTICAL_MILE = 50

# For checking if vessel moves
REQUIRED_MOVE_DIST_M = 100


# Collision detecting thresholds. Minimum required time for collision
COLLISION_TIME_WINDOW_MINUTES = 10

# For finding grid cells
GRID = 0.01

# later
EARTH_RADIUS_METERS = 6371000
EARTH_RADIUS_NM = 3440.065
MIN_HEADING_DIFF = 30


<h3>Big data analytics Task 4</h3>

In [4]:
data_source_folder = "aisdk-2021-12"
paths = [
    os.path.join(
        data_source_folder, 
        data_source_folder + "-" + str(i).rjust(2, "0") + ".csv"
    )
    for i in range(1, 32)
]

print("Using file names")
print(paths[:3])
print("....")

Using file names
['aisdk-2021-12/aisdk-2021-12-01.csv', 'aisdk-2021-12/aisdk-2021-12-02.csv', 'aisdk-2021-12/aisdk-2021-12-03.csv']
....


In [5]:
# Starting new Spark session or getting existing one 
try:
    spark = SparkSession.builder.appName("task_4_cluster").getOrCreate()

    # Test communication with Spark
    spark.range(1).count()

    print(f"Task 4 Spark cluster working. Version = {spark.version}")

except Exception as error:
    print(f"ERROR: {error}")

Task 4 Spark cluster working. Version = 3.5.0


<h3>Data cleaning and pre-processing</h3>

In [6]:
start_time = datetime.now()
start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
print(f"Importing data to PySpark cluster at start time: {start_time_str}")


# Loading data from .csv files
data = spark.read.csv(
    paths,
    sep=",",
    header=True,
    inferSchema=True
)

df_filtered = (
    data
    .select("# Timestamp", "MMSI", "Latitude", "Longitude", "Name", "SOG", "Heading")
    .filter(
        F.col("# Timestamp").isNotNull() &
        F.col("MMSI").isNotNull() &
        F.col("Latitude").isNotNull() &
        F.col("Longitude").isNotNull() &
        F.col("Name").isNotNull() &
        F.col("SOG").isNotNull() &
        F.col("Heading").isNotNull()
    )
)

print("SUCCESS: Import finished and empty line filtering finished")

end_time = datetime.now()
end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")

time_diff_seconds = (end_time - start_time).seconds

print("----------------------")
print(f"Total {time_diff_seconds} seconds")
print(f"Execution end-time: {end_time_str}")


Importing data to PySpark cluster at start time: 2026-06-06 23:51:10
SUCCESS: Import finished and empty line filtering finished
----------------------
Total 207 seconds
Execution end-time: 2026-06-06 23:54:37


In [7]:
# Validating if file read is successfull

read_record_count = df_filtered.count()
print(f"Read records {read_record_count} from files .csv")

Read records 245158653 from files .csv


In [8]:
df_filtered = (
    df_filtered
    .withColumn(
        "timestamp",
        F.to_timestamp(
            F.col("# Timestamp"),
            "dd/MM/yyyy HH:mm:ss"
        )
    )
    .select(
        "timestamp",
        "MMSI",
        "Latitude",
        "Longitude",
        "Name",
        "SOG",
        "Heading"
    )
)

In [9]:
window = Window.partitionBy("MMSI").orderBy("timestamp")

In [10]:
df_filtered = df_filtered.withColumn("rn", row_number().over(window))

In [11]:
# method for calculating distance based on haversine radius
def haversine_in_radius_nm(
    a_lat_col: str, 
    a_lon_col: str, 
    b_lat_value: np.float64, # radius values
    b_lon_value: np.float64 # radius values
):
    phi_2 = radians(a_lat_col)
    phi_1 = radians(lit(b_lat_value))
    lambda_2 = radians(a_lon_col) 
    lambda_1 = radians(lit(b_lon_value))
    
    return (
        EARTH_RADIUS_NM * 2 * asin(
            sqrt(
                sin((phi_2 - phi_1) / 2) ** 2 +
                cos(phi_1) *
                cos(phi_2) *
                sin((lambda_2 - lambda_1) / 2) ** 2
            )
        )
    )

# method calculating distance between two points (Latitude and Longitude)
def haversine_meters(
    a_lat_col: Column,
    a_lon_col: Column,
    b_lat_col: Column,
    b_lon_col: Column
) -> Column:    
    phi_2 = radians(a_lat_col)
    phi_1 = radians(b_lat_col)    
    lambda_2 = radians(a_lon_col)
    lambda_1 = radians(b_lon_col)
    
    return (
        EARTH_RADIUS_METERS * 2 * asin(
            sqrt(
                sin((phi_2 - phi_1) / 2) ** 2 +
                cos(phi_1) *
                cos(phi_2) *
                sin((lambda_2 - lambda_1) / 2) ** 2
            )
        )
    )

In [12]:
CENTER_POINT = Point(
    latitude=55.225000,
    longitude=14.245000
)
print(f"Analyzing ships near radius: {CENTER_POINT}")

Analyzing ships near radius: Point(latitude=55.225, longitude=14.245)


In [13]:
# filter out 50 neutilon miles
df_filtered = df_filtered.filter(
    haversine_in_radius_nm(
        "Latitude", 
        "Longitude",
        CENTER_POINT.latitude, 
        CENTER_POINT.longitude
    ) <= NAUTICAL_MILE
)

In [14]:
ship_records = df_filtered.count()
print(f"Found ships records {ship_records} near radius point: {CENTER_POINT}")

Found ships records 23420392 near radius point: Point(latitude=55.225, longitude=14.245)


In [15]:
# Finding vessels, which are moving SOG > 1 and position 
# Lat and Lon changes based on haversine distance

# Filtering out moving ships
df_filtered = df_filtered.filter(
    col("SOG") > SOG_MOVE
)

# Finding record pair distances in groups
df_filtered = (
    df_filtered
    .withColumn("prev_lat", lag("Latitude").over(window))
    .withColumn("prev_lon", lag("Longitude").over(window))
    .withColumn(
        "move_dist_m",
        haversine_meters(
            col("prev_lat"),
            col("prev_lon"),
            col("Latitude"),
            col("Longitude")
        )
    )
)

# Filtering if vessel moved overall atleast REQUIRED_MOVE_DIST_M meters moved
moving_vessels = (
    df_filtered
    .groupBy("MMSI")
    .agg(
        F.sum(
            F.coalesce(col("move_dist_m"), F.lit(0))
        ).alias("total_move_m")
    )
    .filter(col("total_move_m") > REQUIRED_MOVE_DIST_M)  
    .select("MMSI")
)

# Selecting only moved vessels based on moving vessels distance
df_filtered = (
    df_filtered
    .join(moving_vessels, "MMSI", "inner")
    .drop("prev_lat", "prev_lon", "move_dist_m")
)

In [16]:
moving_ships = df_filtered.count()
print(f"Selected moving ships (SOG > 1 and moved_dist > 0): {moving_ships}")

Selected moving ships (SOG > 1 and moved_dist > 0): 18371705


<h3>Collision analysis</h3>

In [17]:
# Forming data buckets for efficient join operations

df_filtered = (
    df_filtered
    .withColumn("lat_bucket", F.floor(F.col("Latitude") / GRID))
    .withColumn("lon_bucket", F.floor(F.col("Longitude") / GRID))
    .withColumn(
        "time_bucket",
        F.floor(
            F.unix_timestamp("timestamp") / 60
        )
    )
)

In [18]:
# Forming alias based tables for join operation required for comparisons

df_filtered_a = df_filtered.select(
    "timestamp",
    "Name",
    "MMSI",
    "Latitude",
    "Longitude",
    "SOG",
    "Heading",
    "lat_bucket",
    "lon_bucket",
    "time_bucket"
).alias("a")

df_filtered_b = df_filtered.select(
    "timestamp",
    "Name",
    "MMSI",
    "Latitude",
    "Longitude",
    "SOG",
    "Heading",
    "lat_bucket",
    "lon_bucket",
    "time_bucket"
).alias("b")

In [19]:
# Finding all possible ships pairs based on buckets
candidate_pairs = (
    df_filtered_a
    .join(
        df_filtered_b,
        (col("a.MMSI") < col("b.MMSI"))
        &
        (col("a.time_bucket") == col("b.time_bucket"))
        &
        (col("a.lat_bucket") == col("b.lat_bucket"))
        &
        (col("a.lon_bucket") == col("b.lon_bucket"))
    )
    .select(
        col("a.MMSI").alias("mmsi_a"),
        col("a.timestamp").alias("timestamp_a"),
        col("a.Name").alias("name_a"),
        col("a.Latitude").alias("lat_a"),
        col("a.Longitude").alias("lon_a"),
        col("a.SOG").alias("sog_a"),
        col("a.Heading").alias("heading_a"),
        col("b.MMSI").alias("mmsi_b"),
        col("b.timestamp").alias("timestamp_b"),
        col("b.Name").alias("name_b"),
        col("b.Latitude").alias("lat_b"),
        col("b.Longitude").alias("lon_b"),
        col("b.SOG").alias("sog_b"),
        col("b.Heading").alias("heading_b")
    )
)
candidate_pairs = candidate_pairs.withColumn(
    "heading_diff",
    F.least(
        F.abs(F.col("heading_a") - F.col("heading_b")),
        360 - F.abs(F.col("heading_a") - F.col("heading_b"))
    )
)
candidate_pairs = candidate_pairs.filter(
    F.col("heading_diff") > MIN_HEADING_DIFF
)

In [20]:
# Finding possible collision point from which collect all the required points that lead to the ship collision
dist_df = candidate_pairs.withColumn(
    "distance_m",
    haversine_meters(
        col("lat_a"), col("lon_a"),
        col("lat_b"), col("lon_b")
    )
)

min_dist = dist_df.agg(F.min("distance_m")).first()[0]

closest_pair = (
    dist_df
    .filter(F.col("distance_m") == min_dist)
    .first()
)

In [21]:
closest_pair_dict = closest_pair.asDict()
print(f"Ship pair (Ship A, Ship B), which is close to collision:\n")
for key, value in closest_pair_dict.items():
    print(f"{key} : {value}")

Ship pair (Ship A, Ship B), which is close to collision:

mmsi_a : 219019287
timestamp_a : 2021-12-03 16:02:32
name_a : HG 162 NORTH OCEAN
lat_a : 55.243472
lon_a : 15.087747
sog_a : 5.4
heading_a : 220
mmsi_b : 219021428
timestamp_b : 2021-12-03 16:02:54
name_b : HG 165 SOUTH OCEAN
lat_b : 55.243482
lon_b : 15.087745
sog_b : 1.6
heading_b : 173
heading_diff : 47
distance_m : 1.1191536635713002


In [ ]:
print("Selecting ship A path to and after near collision point")
path_a = df_filtered.filter(
    (F.col("MMSI") == closest_pair_dict["mmsi_a"])
    &
    (
        F.abs(
            F.unix_timestamp("timestamp")
            - F.unix_timestamp(F.lit(closest_pair_dict["timestamp_a"]))
        ) <= COLLISION_TIME_WINDOW_MINUTES * 60 
    )
)
path_a.show(truncate=False)

Showing selected ship A path to and after collision
+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|MMSI     |timestamp          |Latitude |Longitude|Name              |SOG|Heading|rn   |lat_bucket|lon_bucket|time_bucket|
+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|219019287|2021-12-03 15:52:32|55.248097|15.088783|HG 162 NORTH OCEAN|1.6|157    |21426|5524      |1508      |27309112   |
|219019287|2021-12-03 15:52:35|55.248077|15.088792|HG 162 NORTH OCEAN|1.6|164    |21427|5524      |1508      |27309112   |
|219019287|2021-12-03 15:52:38|55.248062|15.088802|HG 162 NORTH OCEAN|1.9|167    |21428|5524      |1508      |27309112   |
|219019287|2021-12-03 15:52:41|55.248032|15.088803|HG 162 NORTH OCEAN|2.3|169    |21429|5524      |1508      |27309112   |
|219019287|2021-12-03 15:52:45|55.24799 |15.088822|HG 162 NORTH OCEAN|2.7|168    |21430

In [ ]:
print("Selecting ship B path to and after near collision point")
path_b = df_filtered.filter(
    (F.col("MMSI") == closest_pair_dict["mmsi_b"])
    &
    (
        F.abs(
            F.unix_timestamp("timestamp")
            - F.unix_timestamp(F.lit(closest_pair_dict["timestamp_b"]))
        ) <= COLLISION_TIME_WINDOW_MINUTES * 60 
    )
)
path_b.show(10, truncate=False)

Showing selected ship B path to and after collision
+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|MMSI     |timestamp          |Latitude |Longitude|Name              |SOG|Heading|rn   |lat_bucket|lon_bucket|time_bucket|
+---------+-------------------+---------+---------+------------------+---+-------+-----+----------+----------+-----------+
|219021428|2021-12-03 15:52:54|55.24762 |15.088092|HG 165 SOUTH OCEAN|2.4|166    |22624|5524      |1508      |27309112   |
|219021428|2021-12-03 15:52:56|55.247595|15.0881  |HG 165 SOUTH OCEAN|2.4|166    |22625|5524      |1508      |27309112   |
|219021428|2021-12-03 15:53:01|55.247532|15.08814 |HG 165 SOUTH OCEAN|2.4|164    |22626|5524      |1508      |27309113   |
|219021428|2021-12-03 15:53:04|55.247505|15.088155|HG 165 SOUTH OCEAN|2.4|163    |22627|5524      |1508      |27309113   |
|219021428|2021-12-03 15:53:06|55.247468|15.08818 |HG 165 SOUTH OCEAN|2.4|164    |22628

<h3>Results export to csv</h3>

In [24]:
path_a.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("ship_a_collision")
    
print("Exported Ship A collision data to ship_a_collision folder")

Exported Ship A collision data to ship_a_collision folder


In [25]:
path_b.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("ship_b_collision")
    
print("Exported Ship B collision data to ship_b_collision folder")

Exported Ship B collision data to ship_b_collision folder


In [26]:
spark.stop()